# Parallel Lensless Dataset Tutorial
This Jupyter notebook contains a tutorial for using the dataset from the [ConvRML project](https://lakabuli.github.io/ConvRML/). All data can be found in and downloaded from [this folder](https://drive.google.com/drive/folders/1CqPliG5rZIYH5zA6cXdrA6cvDF7obbvc?usp=drive_link). This notebook provides guidance using 4x downsampled and undistorted ground truth measurements that are warped to each lensless imager's coordinate spaces.

The steps included in this notebook are:
1. Load and process ground truth measurements
2. Load and process lensless imager measurements
3. Pre-process for training
4. Crop for training loss

Acronyms:
- Parallel Lensless Dataset (PLD)
- Ground Truth (GT)
- Random Multi-Focal Lenslet (RML)
- Diffuser (DC)
- Field-of-View (FOV)

This notebook assumes that filenames are not changed within the downloaded data directory structure, and that all files are in a root directory. Please update the root directory path below:

In [15]:
ROOT_DIR = '' # TODO: Add the root directory

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from skimage.transform import rescale, resize
import torch
import kornia.geometry.transform as transform
import tifffile

## 1. Load and process ground truth measurements
We have prepared [a folder](https://drive.google.com/drive/folders/1KeAL_V-usG69LyrqTVabK8Maxa-MLb-R?usp=drive_link) with pre-processed measurements. Specifically, we will be referencing the `4x Downsampled GT2RML` and `4x Downsampled GT2DC` folders. Starting with the full-resolution (1200, 1920) measurements, these ground truth measurements have been undistorted to correct for lens distortion, downsampled by a factor of 4 to form (300, 480) measurements, and warped to the space of each respective lensless imager.

This section performs the following operations:
- loads a single ground truth image from the dataset with shape (300, 480, 4)
- removes the alpha channel associated with `matplotlib.imread`

Download and unzip the files:
- [4x GT to RML](https://drive.google.com/drive/folders/17An3asIdUsMdJLHZHdcdKLiVQdktMhIt?usp=sharing)
- [4x GT to DC](https://drive.google.com/drive/folders/1BiSZolF2vhpTSJ20N9zPeW_g-6URVjoa?usp=sharing)

### Ground Truth to RML

In [ ]:
## Ground Truth to RML
object_idx = 64
gt2RML_image_path = os.path.join(ROOT_DIR, '4x_undistorted_GT2RML', 'warped_4x_undistorted_img_{}_cam_2.tiff'.format(object_idx))
gt2RML_image = tifffile.imread(gt2RML_image_path)
print("Image shape: ", gt2RML_image.shape) # verify that shape is (300, 480, 4)

plt.figure()
plt.title("GT2RML Image (300, 480)")
plt.imshow(gt2RML_image)

### Ground Truth to DC

In [ ]:
## Ground Truth to Diffuser
object_idx = 64
gt2DC_image_path = os.path.join(ROOT_DIR, 'undistorted_GT2DC', 'warped_4x_undistorted_img_{}_cam_2.tiff'.format(object_idx))
gt2DC_image = tifffile.imread(gt2DC_image_path)
print("Image shape: ", gt2DC_image.shape) # verify that shape is (300, 480, 4)

plt.figure()
plt.title("GT2DC Image (300, 480)")
plt.imshow(gt2DC_image)

## 2. Load and process lensless imager measurements
Lensless imager measurements need to be 4x downsampled to match the undistorted ground truth resolution: (1200, 1920) -> (300, 480).

We provide lensless imager measurements that are already 4x downsampled. You can skip this step if you choose to use the pre-downsampled measurements directly. Simply download and unzip the files, then load the desired measurements.
- [4x downsampled RML measurements](https://drive.google.com/drive/folders/1S4j2xdGOZdZd1igdy5TtqlFkpwaXSbDE?usp=share_link)
- [4x downsampled DC measurements](https://drive.google.com/drive/folders/1S4j2xdGOZdZd1igdy5TtqlFkpwaXSbDE?usp=share_link)

---
If you would like to downsample yourself or switch to a different downsample factor, download and unzip the full resolution measurements:
- [Full resolution RML measurements](https://drive.google.com/drive/folders/1G_60zOp4hplFz-JzVHVLHieTEuO3Ykjx?usp=sharing)
- [Full resolution DC measurements](https://drive.google.com/drive/folders/1IoCGBQjfTSus04yUnBvpPbQZrOAnGJ7J?usp=sharing).


### Downsample RML measurements

In [ ]:
# Downsampling RML measurements
object_idx = 64
RML_image_path = os.path.join(ROOT_DIR, 'rml', 'img_{}_cam_1.tiff'.format(object_idx))
RML_image = tifffile.imread(RML_image_path)
print("Image shape: ", RML_image.shape) # verify that shape is (1200, 1920, 3)
ds_RML_image = resize(RML_image, (300, 480), anti_aliasing=True).astype(np.float32) # resize to 300 x 480
print("Resized shape: ", RML_image.shape)

plt.figure()
plt.title("Original RML Measurement (1200, 1920)")
plt.imshow(RML_image)

plt.figure()
plt.title("Downsampled RML Measurement (300, 480)")
plt.imshow(ds_RML_image)

### Downsample DC measurements

In [ ]:
## Downsampling DC measurements
object_idx = 64
DC_image_path = os.path.join(ROOT_DIR, 'diffuser', 'img_{}_cam_0.tiff'.format(object_idx))
DC_image = tifffile.imread(DC_image_path)
print("Image shape: ", DC_image.shape) # verify that shape is (1200, 1920, 3)
ds_DC_image = resize(DC_image, (300, 480), anti_aliasing=True).astype(np.float32) # resize to 300 x 480
print("Resized shape: ", DC_image.shape)

plt.figure()
plt.title("Original DC Measurement (1200, 1920)")
plt.imshow(DC_image)

plt.figure()
plt.title("Downsampled DC Measurement (300, 480)")
plt.imshow(ds_DC_image)

## 3. Pre-processing for training
Once ground truth and lensless measurements have been loaded and downsampled, pre-processing may be necessary for training image reconstructions. For our image reconstruction algorithms in Pytorch, images are pre-processed as follows:
- ensure images have 3 channels
- convert from uint8 to float32
- resize image to correct downsampling level (may be redundant but is a good check). We use 4x downsampling (300, 480) but change this based on your downsampling level if you are using other data
- normalize to 0 - 1 and clip image to 0-1
- convert to a torch tensor
- reshape to have channel dimension first (H, W, C) -> (C, H, W)

In this example, we use an RML image to illustrate the process.

In [ ]:
image = ds_RML_image
target = gt2RML_image
downsampled_dims = (300, 480)

In [8]:
# if images include alpha channel, remove
if image.shape[-1] == 4:
    image = image[..., :-1]
if target.shape[-1] == 4:
    target = target[..., :-1]

# convert images to 0 to 1 range and convert to float, they're 8 bit 0 - 255 otherwise 
image = (image / 255.0).astype(np.float32)
target = (target / 255.0).astype(np.float32) 

# resize measurement 
image = resize(image, downsampled_dims, anti_aliasing=True).astype(np.float32)
assert image.shape == target.shape 

# clip 0-1
image = np.clip(image, 0,1)
target = np.clip(target, 0, 1)

image = torch.from_numpy(image)
target = torch.from_numpy(target)

# Move channels to the front
image = torch.moveaxis(image, -1, 0)
target = torch.moveaxis(target, -1, 0)

## 4. Cropping for training loss

In training, you may want to crop the image to remove the black borders before loss evaluation. Since these reconstructions are done in each lensless imager's space, there is a different crop FOV necessary. Below, we use the crop regions of the RML and Diffuser for the 4x downsampled (300, 480) reconstructions used in the ConvRML project. 

The ground truth image is used in this example, but you should replace the image with recons for each respective imager.

#### RML

In [ ]:
# load rml recons 
rml_recon = gt2RML_image # placeholder, replace with your recon
cropped_image = rml_recon[36:266, 133:363, :]
plt.imshow(cropped_image)

#### Diffuser

In [ ]:
# load dc recons 
dc_recon = gt2DC_image # placeholder, replace with your recon
cropped_image = dc_recon[23:280, 112:369, :]
plt.imshow(cropped_image)

## 5. Evaluation view
In order to compare the three imagers (GT, DC, RML) in the same imaging space, we can warp both lensless imagers to the space of the ground truth camera and crop the black borders accordingly. To do this, we use a calibrated homography matrix and the Kornia package. We assume a single image, or batch size of 1, in this example, and that images are saved as `.npy` files.

Download [the homography transform files here.](https://drive.google.com/drive/folders/1hfcoBQc2XNIkmWxK5hOzHYO0GE6Fdfsj?usp=drive_link)
- `RML2GT_homography_4x_2026.torch`
- `DC2GT_homography_4x_2026.torch`

### Path to RML homography and load homography:

In [ ]:
rml_homog_path = os.path.join(ROOT_DIR, 'Lensless2GroundTruth') # Lensless2GroundTruth
rml_homography = torch.load(os.path.join(rml_homog_path, 'RML2GT_homography_4x_2026.torch'))

### Load RML recons and apply homography

In [ ]:
# recons need to be in b x c x h x w format 
rml_recon_path = '' # TODO: Add path to your recons
rml_recon = np.load(rml_recon_path)
rml_recon_tensor = torch.from_numpy(rml_recon) 
rml_recon_tensor = transform.warp_perspective(rml_recon_tensor, rml_homography, dsize=(300, 480), align_corners=True)
cropped_rml_recon = rml_recon_tensor[52:266, 129:343, :]
plt.imshow(cropped_rml_recon)

### Repeat for DC

In [ ]:
dc_homog_path = os.path.join(ROOT_DIR, 'Lensless2GroundTruth')
dc_homography = torch.load(os.path.join(dc_homog_path, 'DC2GT_homography_4x_2026.torch'))

In [ ]:
# recons need to be in b x c x h x w format 
dc_recon_path = '' # TODO: Add path to your recons
dc_recon = np.load(dc_recon_path)
dc_recon_tensor = torch.from_numpy(dc_recon) 
dc_recon_tensor = transform.warp_perspective(dc_recon_tensor, dc_homography, dsize=(300, 480), align_corners=True)
cropped_dc_recon = dc_recon_tensor[52:266, 129:343, :]
plt.imshow(cropped_dc_recon)